In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import ArrowStyle
import seaborn as sns
import json
import warnings
import shutil
from tqdm import tqdm
from adjustText import adjust_text
from sklearn.linear_model import LinearRegression
from pathlib import Path
import multiprocessing
import functools
warnings.filterwarnings('ignore')
%config InlineBackend.figure_format = 'retina'

### Load Data Functions ###

In [ ]:
# keys are filenames
gt_df = pd.read_csv('../data/imagenet/ann/gt.csv', index_col=0)

def add_occ_frac(cfg):
    '''
    Compute occlusion fraction and add it config.
    '''
    occ_transform = cfg['occ_transform']
    if occ_transform == 'patch':
        cfg['occ_frac'] = cfg['npatch_occ'] / (cfg['npatch_h'] * cfg['npatch_w'])
    elif occ_transform == 'bar':
        cfg['occ_frac'] = cfg['nbar_occ'] / cfg['nbar']
    elif occ_transform == 'simplex_noise':
        npix_occ = cfg['npix_occ']
        if isinstance(npix_occ, int):
            cfg['occ_frac'] = npix_occ / (224 * 224)
        else:
            cfg['occ_frac'] = tuple(e / (224 * 224) for e in npix_occ)
    elif occ_transform == 'empty':
        cfg['occ_frac'] = 0
    elif occ_transform == 'full':
        cfg['occ_frac'] = 1
    else:
        raise ValueError(f'Invalid occlusion transform type: {occ_transform}.')
        
def lst_to_tup(cfg):
    '''
    Convert list dictionary values to tuples, so they 
    are hashable by DF.
    '''
    keys = ['npix_occ', 'noise_freq']
    for key in keys:
        if key not in cfg:
            continue
        if isinstance(cfg[key], list):
            cfg[key] = tuple(cfg[key])

def mk_cfg(exp_subdir, models, exp, include_ims):
    '''
    Helper for mk_df.
    '''
    cfg_lst = []
    # get cfg and acc
    cfg_loc = exp_subdir / 'cfg.json'
    stat_loc = exp_subdir / 'stat.json'
    if not stat_loc.exists():
        return []
    with open(cfg_loc, 'r') as f:
        cfg = json.load(f)
    if models is not None and cfg['model'] not in models:
        return []
    with open(stat_loc, 'r') as f:
        stat = json.load(f)
    # if inpainting model used in precomputed images, then locate
    # occlusion parameters of the precomputed image
    if 'inpaint_model_precomp' in cfg and cfg['inpaint_model_precomp'] is not None:
        subexp_name = cfg['data_dir'].split('/')[-1]
        orig_cfg_loc = Path(f'../data/human_occ_exp') / exp / subexp_name / 'cfg.json'
        with open(orig_cfg_loc, 'r') as f:
            orig_cfg = json.load(f)
        cfg |= orig_cfg
    # if occ_transform is 'precomputed', populate the 
    # missing transform-related fields like occ_frac
    if 'occ_transform' in cfg and cfg['occ_transform'] == 'precomputed':
        subexp_name = Path(cfg['mask_path']).parent.name
        orig_cfg_loc = Path(f'../data/human_occ_exp') / exp / subexp_name / 'cfg.json'
        with open(orig_cfg_loc, 'r') as f:
            orig_cfg = json.load(f)
        del cfg['occ_transform'] # will be replaced by value in orig_cfg
        cfg |= orig_cfg
    lst_to_tup(cfg)
    add_occ_frac(cfg)
    cfg = cfg | stat
    # get finetuning parameters for finetuned experiments
    if 'checkpoint' in cfg and cfg['checkpoint'] != 'pretrained':
        ft_exp_subdir = Path(cfg['checkpoint']).parent
        ft_cfg_loc = ft_exp_subdir / 'cfg.json'
        with open(ft_cfg_loc, 'r') as f:
            ft_cfg = json.load(f)
        lst_to_tup(ft_cfg)
        add_occ_frac(ft_cfg)
        # add suffix to finetuning config params to avoid collisions with
        # validation cfg params
        new_ft_cfg = {}
        for k, v in ft_cfg.items():
            new_ft_cfg[f'{k}_ft'] = v
        cfg |= new_ft_cfg
    # include each image rather than mean accuracy if desired
    if include_ims:
        pred_loc = exp_subdir / 'pred.csv'
        pd_df = pd.read_csv(pred_loc)
        # top1 key corresponds to entire experiment, not this image
        del cfg['top1']
        for i in pd_df.index:
            new_cfg = cfg.copy()
            fn = pd_df.loc[i].filename
            pd_idx = pd_df.loc[i].pd_idx
            gt_idx = gt_df.loc[fn].gt_idx
            correct = pd_idx == gt_idx
            new_cfg |= {
                'filename': fn,
                'pd_idx': pd_idx,
                'gt_idx': gt_idx,
                'correct': correct,
            }
            # separate config for each image
            cfg_lst.append(new_cfg)
    else:
        # one config for experiment
        cfg_lst.append(cfg)
    return cfg_lst

def mk_df(exp, include_ims=False, models=None):
    '''
    Make big DF with all eval results for one experiment.
    '''
    
    exp_dir = Path(f'../results/{exp}/eval')
    # create DF with accuracy and params for each occlusion config
    exp_subdirs = sorted(exp_dir.glob('*'))
    partial_mk_cfg = functools.partial(mk_cfg, models=models, exp=exp, include_ims=include_ims)
    with multiprocessing.Pool(50) as pool:
        results = pool.map(partial_mk_cfg, exp_subdirs)
    cfgs = []
    for cfg_lst in results:
        cfgs += cfg_lst
    df = pd.DataFrame(cfgs)
    return df

### Figure Functions ###

In [ ]:
pal = sns.color_palette('tab10')

def single_acc_vs_freq(df, name, save_name=None):
    fig, ax = plt.subplots(constrained_layout=True)
    fig.set_size_inches(7, 5)
    ax.set_title(f'Accuracy vs. occlusion frequency ({name})', fontsize=15)
    df2 = df[df.occ_transform == 'simplex_noise']
    df2['noise_freq'] = df2.noise_freq.astype(int).astype('category')
    df2['occ_frac'] = df2.occ_frac.astype('category')
    plot = sns.lineplot(data=df2, x='noise_freq', y='top1', hue='occ_frac', marker='o', ax=ax, errorbar='ci', err_style='bars', palette=pal)
    ax.legend(loc='lower left', title='Occlusion fraction')
    ax.set_xlabel('Occlusion frequency', fontsize=15)
    ax.set_ylabel('Accuracy', fontsize=15)
    ax.set_xscale('log')
    ax.set_ylim(-0.05, 1.05)
    if save_name is None:
        save_name = name
    fig.savefig(f'out/single_acc_vs_freq_{save_name}.png', dpi=100)
    plt.show()
    
def single_acc_vs_occ_frac(df, name, save_name=None):
    fig, ax = plt.subplots(constrained_layout=True)
    fig.set_size_inches(7, 5)
    ax.set_title(f'Accuracy vs. occlusion fraction ({name})', fontsize=15)
    df2 = df[df.occ_transform == 'simplex_noise']
    df2['noise_freq'] = df2.noise_freq.astype(int).astype('category')
    df2['occ_frac'] = df2.occ_frac.astype('category')
    plot = sns.lineplot(data=df2, x='occ_frac', y='top1', hue='noise_freq', marker='o', ax=ax, errorbar='ci', err_style='bars', palette=pal)
    ax.legend(loc='lower left', title='Occlusion frequency')
    ax.set_xlabel('Occlusion fraction', fontsize=15)
    ax.set_ylabel('Accuracy', fontsize=15)
    ax.set_ylim(-0.05, 1.05)
    if save_name is None:
        save_name = name
    fig.savefig(f'out/single_acc_vs_occ_frac_{save_name}.png', dpi=100)
    plt.show()
    

def sample_models_acc_vs_freq(df, name, save_name=None):
    ncol = 3
    nrow = round(np.ceil(len(sample_hr_models) // ncol))
    fig, axs = plt.subplots(nrow, ncol, constrained_layout=True)
    fig.set_size_inches(ncol * 7, nrow * 5)
    fig.suptitle(f'Accuracy vs. occlusion frequency for various models ({name})', fontsize=20)
    for idx, hr_model in enumerate(sample_hr_models):
        i, j = idx // 3, idx % 3
        ax = axs[i, j]
        df2 = df[(df.hr_model == hr_model) & (df.occ_transform == 'simplex_noise')]
        df2['noise_freq'] = df2.noise_freq.astype(int).astype('category')
        df2['occ_frac'] = df2.occ_frac.astype('category')
        plot = sns.lineplot(data=df2, x='noise_freq', y='top1', hue='occ_frac', marker='o', ax=ax, errorbar='ci', err_style='bars')
        ax.legend(loc='lower left', title='Occlusion fraction')
        ax.set_xlabel('Occlusion frequency', fontsize=20)
        ax.set_ylabel('Accuracy', fontsize=20)
        ax.set_xscale('log')
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(hr_model, fontsize=20)
    if save_name is None:
        save_name = name
    fig.savefig(f'out/sample_models_acc_vs_freq_{save_name}.png', dpi=100)
    plt.show()

def sample_models_acc_vs_occ_frac(df, name, save_name=None):
    ncol = 3
    nrow = round(np.ceil(len(sample_hr_models) // ncol))
    fig, axs = plt.subplots(nrow, ncol, constrained_layout=True)
    fig.set_size_inches(ncol * 7, nrow * 5)
    fig.suptitle(f'Accuracy vs. occlusion fraction for various models ({name})', fontsize=20)
    for idx, hr_model in enumerate(sample_hr_models):
        i, j = idx // 3, idx % 3
        ax = axs[i, j]
        df2 = df[(df.hr_model == hr_model) & (df.occ_transform == 'simplex_noise')]
        df2['noise_freq'] = df2.noise_freq.astype('int').astype('category')
        df2['occ_frac'] = df2.occ_frac.astype('category')
        plot = sns.lineplot(data=df2, x='occ_frac', y='top1', hue='noise_freq', marker='o', ax=ax, errorbar='ci', err_style='bars')
        ax.legend(loc='lower left', title='Occlusion frequency')
        ax.set_xlabel('Occlusion fraction', fontsize=20)
        ax.set_ylabel('Accuracy', fontsize=20)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(hr_model, fontsize=20)
    if save_name is None:
        save_name = name
    fig.savefig(f'out/sample_models_acc_vs_occ_frac_{save_name}.png', dpi=100)
    plt.show()
    
def acc_occ_bar(df, name, save_name=None):
    means = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac > 0) & (df.occ_frac < 1)].groupby('hr_model').top1.mean().sort_values(ascending=False)
    plt.figure(figsize=(14, 6))
    sns.barplot(x=means.index, y=means)
    plt.xticks(rotation=90)
    plt.ylim(0, 1)
    plt.title(f'Occluded accuracy by model ({name})', fontsize=15)
    plt.xlabel('Model', fontsize=15)
    plt.ylabel('Accuracy', fontsize=15)
    plt.tight_layout()
    if save_name is None:
        save_name = name
    plt.savefig(f'out/acc_occ_bar_{save_name}.png', dpi=100)
    plt.show()

def acc_unocc_bar(df, name, save_name=None):
    means = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac == 0)].groupby('hr_model').top1.mean().sort_values(ascending=False)
    plt.figure(figsize=(14, 6))
    sns.barplot(x=means.index, y=means)
    plt.xticks(rotation=90)
    plt.ylim(0, 1)
    plt.title(f'Unoccluded accuracy by model ({name})', fontsize=15)
    plt.xlabel('Model', fontsize=15)
    plt.ylabel('Accuracy', fontsize=15)
    plt.tight_layout()
    if save_name is None:
        save_name = name
    plt.savefig(f'out/acc_unocc_bar_{save_name}.png', dpi=100)
    plt.show()
    
def acc_unocc_occ_line(dfs, names, save_names=None):
    plt.figure(figsize=(14, 6))
    # get the colormap
    cmap = plt.cm.get_cmap('Paired')
    # only keep models in all dfs
    hr_models_set = set(dfs[0].hr_model.unique())
    for df in dfs:
        hr_models_set = hr_models_set.intersection(set(df.hr_model.unique()))
    # accuracy for each df
    for i, (df, name) in enumerate(zip(dfs, names)):
        c1 = cmap(2 * i)
        c2 = cmap(2 * i + 1)
        df = df[df.hr_model.isin(hr_models_set)]
        unocc_means = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac == 0)].groupby('hr_model').top1.mean().sort_values(ascending=False)
        sns.lineplot(x=unocc_means.index, y=unocc_means, label=f'Unoccluded ({name})', marker='o', color=c1, linestyle=':')
        occ_means = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac > 0)].groupby('hr_model').top1.mean().sort_values(ascending=False)
        sns.lineplot(x=occ_means.index, y=occ_means, label=f'Occluded ({name})', marker='o', color=c2)
    plt.xticks(rotation=90)
    plt.ylim(0, 1)
    plt.title(f'Accuracy by model', fontsize=15)
    plt.xlabel('Model', fontsize=15)
    plt.ylabel('Accuracy', fontsize=15)
    plt.tight_layout()
    if save_names is None:
        save_names = names
        save_names_str = '_'.join(save_names)
    plt.savefig(f'out/acc_unocc_occ_line_{save_names_str}.png', dpi=100)
    plt.show()
    
def acc_occ_vs_unocc_scatterplot(df, occ_frac_range, occ_frac_name, name, save_name=None):
    occ_frac_lo, occ_frac_hi = occ_frac_range
    occ_df = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac >= occ_frac_lo) & (df.occ_frac <= occ_frac_hi)]
    mean_occ_acc = occ_df.groupby('model').top1.mean().sort_values(ascending=False)
    unocc_df = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac == 0)]
    unocc_acc = unocc_df.groupby('model').top1.mean().sort_values(ascending=False)
    # get the colormap
    cmap = plt.cm.get_cmap('tab20')
    # instantiate graphs
    fig, axs = plt.subplots(1, 2, figsize=(20, 10))
    fig.suptitle(f'Occluded vs. unoccluded accuracy ({name}, {occ_frac_name})', fontsize=20)
    axs[0].set_title('Zoomed out view', fontsize=20)
    axs[1].set_title('Zoomed in view', fontsize=20)
    # draw each graph
    for i, ax in enumerate(axs):
        # keep track of all texts
        texts = []
        repel_xs = []
        repel_ys = []
        for j, (model_fam_name, model_fam) in enumerate(model_dict.items()):
            # color
            c = cmap(j % 20)
            # only plot models present in DF
            model_fam = [model for model in model_fam if model in unocc_acc]
            if len(model_fam) == 0:
                continue
            # get model family points
            unocc_acc_fam = unocc_acc.loc[model_fam]
            mean_occ_acc_fam = mean_occ_acc.loc[model_fam]
            # draw points on graph
            sns.lineplot(x=unocc_acc_fam, y=mean_occ_acc_fam, label=model_fam_name, color=c, marker='o', ax=ax, sort=False)
            # add labels
            x_prev = None
            y_prev = None
            for model in model_fam:
                x = unocc_acc_fam.loc[model]
                y = mean_occ_acc_fam.loc[model]
                hr_model = model_to_hr_model[model]
                hr_suff = model_to_hr_suff[model]
                if i == 1:
                    texts.append(ax.text(x, y, hr_suff))
                # add repel points and arrows
                if x_prev is not None and y_prev is not None:
                    x_mid = (x + x_prev) / 2
                    y_mid = (y + y_prev) / 2
                    x_diff = x - x_prev
                    y_diff = y - y_prev
                    norm = np.sqrt(x_diff**2 + y_diff**2)
                    dx = x_diff / norm * 1e-3
                    dy = y_diff / norm * 1e-3
                    ax.annotate('', xytext=(x_mid, y_mid), xy=(x_mid + dx, y_mid + dy), arrowprops=dict(arrowstyle='-|>', color=c))
                    if i == 1:
                        repel_xs += np.linspace(x_prev, x, 100).tolist()
                        repel_ys += np.linspace(y_prev, y, 100).tolist()
                x_prev = x
                y_prev = y

        # adjust texts, so labels don't overlap
        if i == 1:
            mult = 1
            adjust_text(
                texts, repel_xs, repel_ys,
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=1),
                force_text=(0.1*mult, 0.25*mult), force_points=(0.2*mult, 0.5*mult),
                ax=ax,
            )
        # draw lines on one of the graphs
        if i == 0:
            lr = LinearRegression(fit_intercept=True)
            lr.fit(np.array([unocc_acc]).T, mean_occ_acc)
            m, b = lr.coef_[0], lr.intercept_
            ax.plot([0, 1], [b, m + b], linestyle='--', color='red')
            ax.plot([0, 1], [0, 1], linestyle='--', color='black')
        # other graph stuff
        ax.legend(title='Model family')
        if i == 0:
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
        ax.set_xlabel('Unoccluded accuracy', fontsize=20)
        ax.set_ylabel('Occluded accuracy', fontsize=20)
        fig.tight_layout()
    if save_name is None:
        save_name = name
    plt.savefig(f'out/acc_occ_vs_unocc_scatterplot_{occ_frac_lo}_{occ_frac_hi}_{save_name}.png', dpi=100)
    plt.show()
    
def acc_ratio_vs_num_param_scatterplot(df, occ_frac_range, occ_frac_name, name, save_name=None):
    occ_frac_lo, occ_frac_hi = occ_frac_range
    occ_df = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac >= occ_frac_lo) & (df.occ_frac <= occ_frac_hi)]
    mean_occ_acc = occ_df.groupby('model').top1.mean().sort_values(ascending=False)
    unocc_df = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac == 0)]
    unocc_acc = unocc_df.groupby('model').top1.mean().sort_values(ascending=False)
    # occ to unocc accuracy ratio
    acc_ratio = mean_occ_acc / unocc_acc
    # num param
    num_param = occ_df.groupby('model').apply(lambda df: df.num_param.unique()[0] / 1e6)
    # get the colormap
    cmap = plt.cm.get_cmap('tab20')
    # instantiate graphs
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    title = f'Occlusion accuracy ratio vs. num. of param. ({name}, {occ_frac_name})'
    fig.suptitle(title, fontsize=15)
    xscale = 'log'
    ax.set_xscale(xscale)
    # keep track of all texts
    texts = []
    repel_xs = []
    repel_ys = []
    for j, (model_fam_name, model_fam) in enumerate(model_dict.items()):
        # if in1k_train is True:
        #     if not '(IN1k)' in model_fam_name:
        #         continue
        # else:
        #     if '(IN1k)' in model_name_fam:
        #         continue
        # color
        c = cmap(j % 20)
        # only plot models present in DF
        model_fam = [model for model in model_fam if model in unocc_acc]
        if len(model_fam) == 0:
            continue
        # get model family points
        acc_ratio_fam = acc_ratio.loc[model_fam]
        num_param_fam = num_param.loc[model_fam]
        # draw points on graph
        sns.lineplot(x=num_param_fam, y=acc_ratio_fam, label=model_fam_name, color=c, marker='o', ax=ax, sort=False)
        # add labels
        x_prev = None
        y_prev = None
        for model in model_fam:
            x = num_param_fam.loc[model]
            y = acc_ratio_fam.loc[model]
            hr_model = model_to_hr_model[model]
            hr_suff = model_to_hr_suff[model]
            texts.append(ax.text(x, y, hr_suff))
            # add repel points and arrows
            if x_prev is not None and y_prev is not None:
                if xscale == 'log':
                    x_mid = np.sqrt(x * x_prev)
                else:
                    x_mid = (x + x_prev) / 2
                y_mid = (y + y_prev) / 2
                x_diff = x - x_prev
                y_diff = y - y_prev
                norm = np.sqrt(x_diff**2 + y_diff**2)
                dx = x_diff / norm * 1e-3
                dy = y_diff / norm * 1e-3
                arrowstyle = ArrowStyle("-|>", head_length=0.3, head_width=0.3)
                ax.annotate('', xytext=(x_mid, y_mid), xy=(x_mid + dx, y_mid + dy), arrowprops=dict(arrowstyle=arrowstyle, color=c))
                if xscale == 'log':
                    repel_xs += np.exp(np.linspace(np.log(x_prev), np.log(x), 100)).tolist()
                else:
                    repel_xs += np.linspace(x_prev, x, 100).tolist()
                repel_ys += np.linspace(y_prev, y, 100).tolist()
            x_prev = x
            y_prev = y

    # adjust texts, so labels don't overlap
    mult = 1
    adjust_text(
        texts, repel_xs, repel_ys,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5, alpha=1),
        force_text=(0.1*mult, 0.25*mult), force_points=(0.2*mult, 0.5*mult),
        ax=ax,
    )
    ax.legend(title='Model family')
    ax.set_xlabel('Number of parameters (millions)', fontsize=15)
    ax.set_ylabel('Ratio of occluded to unoccluded accuracy', fontsize=15)
    fig.tight_layout()
    if save_name is None:
        save_name = name
    plt.savefig(f'out/acc_ratio_vs_num_param_scatterplot_{occ_frac_lo}_{occ_frac_hi}_{save_name}.png', dpi=100)
    plt.show()

### Get Model Info ###

In [ ]:
# get model names
models_loc = Path('../scripts/models.json')
with open(models_loc, 'r') as f:
    model_hr_suff_dict = json.load(f)

# make model dict (with no suffixes)
model_dict = {}
for model_fam_name, model_fam in model_hr_suff_dict.items():
    model_dict[model_fam_name] = []
    for model, _ in model_fam:
        model_dict[model_fam_name].append(model)

# make dict that maps timm model names to human-readable model names
model_to_hr_model = {}
model_to_hr_suff = {}
for model_fam_name, model_fam in model_hr_suff_dict.items():
    # parse dataset from model family name 
    model_fam_name, ds = model_fam_name.split()
    # make human readable name for each model
    for model, hr_suff in model_fam:
        hr_model = f'{model_fam_name}-{hr_suff} {ds}'
        model_to_hr_model[model] = hr_model
        model_to_hr_suff[model] = hr_suff
        
# sample models
sample_hr_models = [
    'VGG-16 (IN1k)', 'ResNet-50 (IN1k)', 'ConvNeXt-B (IN1k)',
    'DeiT-B (IN1k)', 'Swin-B (IN21k)', 'CLIP-B/16 (WIT)',
]

sample_model_fams = [
    'VGG (IN1k)', 'ResNet (IN1k)', 'ConvNeXt (IN1k)',
    'ConvNeXt-v2 (IN1k)', 'ViT (IN21k)', 'MAE (IN1k)', 
    'DeiT (IN1k)', 'Swin (IN1k)', 'Swin (IN21k)', 'CLIP (WIT)',
]
model_subset = []
for model_fam in sample_model_fams:
    model_subset += model_dict[model_fam]

### Create Figs for Pretrained and Finetuned Experiments ###

In [ ]:
occ_fracs = [i / 8 for i in range(0, 8)]
noise_freqs = [2**i for i in range(0, 9)]
occ_frac_ranges = [(0.125, 0.875)]
occ_frac_names = ['all occ. fractions']
nocc = len(occ_fracs) * len(noise_freqs)

# load pretrained data
pt_name = 'pretrained'
pt_df = mk_df('exp0.9')
pt_df['hr_model'] = pt_df.model.map(model_to_hr_model)
pt_df = pt_df[pt_df.occ_frac.isin(occ_fracs) & pt_df.noise_freq.isin(noise_freqs)] 

# load finetuned data
ft_name = 'finetuned'
ft_df = mk_df('exp1.1')
ft_df['hr_model'] = ft_df.model.map(model_to_hr_model) 
ft_df = ft_df[ft_df.occ_frac.isin(occ_fracs) & ft_df.noise_freq.isin(noise_freqs)] 
ft_df = ft_df[(ft_df.occ_frac_ft == tuple(occ_fracs)) & (ft_df.noise_freq_ft == tuple(noise_freqs))]

assert pt_df.groupby('model').top1.count().unique().item() == nocc
assert ft_df.groupby('model').top1.count().unique().item() == nocc
print(pt_df.model.nunique(), ft_df.model.nunique())

In [ ]:
# create pretrained figs
sample_models_acc_vs_freq(df=pt_df, name=pt_name)
sample_models_acc_vs_occ_frac(df=pt_df, name=pt_name)

acc_unocc_occ_line(dfs=[pt_df], names=[pt_name])

for occ_frac_range, occ_frac_name in zip(occ_frac_ranges, occ_frac_names):
    acc_occ_vs_unocc_scatterplot(df=pt_df, occ_frac_range=occ_frac_range, name=pt_name, occ_frac_name=occ_frac_name)
    
for occ_frac_range, occ_frac_name in zip(occ_frac_ranges, occ_frac_names):
    acc_ratio_vs_num_param_scatterplot(df=pt_df, occ_frac_range=occ_frac_range, name=pt_name, occ_frac_name=occ_frac_name)

In [ ]:
# create finetuned figs
sample_models_acc_vs_freq(df=ft_df, name=ft_name)
sample_models_acc_vs_occ_frac(df=ft_df, name=ft_name)

acc_unocc_occ_line(dfs=[pt_df, ft_df], names=[pt_name, ft_name])

for occ_frac_range, occ_frac_name in zip(occ_frac_ranges, occ_frac_names):
    acc_occ_vs_unocc_scatterplot(df=ft_df, occ_frac_range=occ_frac_range, name=ft_name, occ_frac_name=occ_frac_name)

for occ_frac_range, occ_frac_name in zip(occ_frac_ranges, occ_frac_names):
    acc_ratio_vs_num_param_scatterplot(df=ft_df, occ_frac_range=occ_frac_range, name=ft_name, occ_frac_name=occ_frac_name)

### Create Figs for Human Dataset Experiments ###

In [ ]:
# functions for results on human dataset

# human noise freqs and occ fracs
hocc_fracs = [0, 0.125, 0.25, 0.5, 0.75, 0.875]
hnoise_freqs = [2**i for i in range(9)]
hnocc = len(hocc_fracs) * len(hnoise_freqs)

cmap = plt.cm.get_cmap('Set2')
    
def load_hdf(path):
    # load human results on human experiment dataset
    human_hdf = pd.read_csv(path)
    # replace empty transforms with simplex noise for easier analysis
    human_hdf = empty_to_simplex_noise(df=human_hdf, occ_fracs=hocc_fracs, noise_freqs=hnoise_freqs)
    # rename 'correct' column to 'top1'
    human_hdf = human_hdf.rename(columns={'correct': 'top1'})
    return human_hdf

def single_compare(col, labels, dfs, name, legend_name=None, save_name=None):
    fig, ax = plt.subplots(constrained_layout=True)
    fig.set_size_inches(7, 5)
    ax.set_ylabel('Accuracy', fontsize=15)
    if col == 'noise_freq':
        ax.set_title(f'Accuracy vs. occlusion frequency ({name})', fontsize=15)
        ax.set_xscale('log')
        ax.set_xlabel('Occlusion frequency', fontsize=15)
    else:
        ax.set_title(f'Accuracy vs. occlusion fraction ({name})', fontsize=15)
        ax.set_xlabel('Occlusion fraction', fontsize=15)
    for i, (label, df) in enumerate(zip(labels, dfs)):
        # don't include occlusion fraction 0 when graphing by frequency
        if col == 'noise_freq':
            df = df[df.occ_frac > 0]
        linestyle = '-'
        sns.lineplot(data=df, x=col, y='top1', marker='o', label=label, color=cmap(i), linestyle=linestyle, ax=ax, err_style='bars', errorbar='ci')
    ax.legend(title=legend_name, loc='lower left')
    if save_name is None:
        save_name = name
    plt.savefig(f'out/single_compare_{col}_{save_name}')
    plt.show()
    
def sample_models_compare(col, labels, dfs, legend_name=None, save_name=None, linestyles=None):
    ncol = 3
    nrow = round(np.ceil(len(sample_hr_models) // ncol))
    fig, axs = plt.subplots(nrow, ncol, constrained_layout=True)
    fig.set_size_inches(ncol * 7, nrow * 5)
    if col == 'noise_freq':
        fig.suptitle(f'Accuracy vs. occlusion frequency for various models', fontsize=20)
    else:
        fig.suptitle(f'Accuracy vs. occlusion fraction for various models', fontsize=20)
    for idx, hr_model in enumerate(sample_hr_models):
        i, j = idx // 3, idx % 3
        ax = axs[i, j]
        for i, (df, label) in enumerate(zip(dfs, labels)):
            # don't include occlusion fraction 0 when graphing by frequency
            if col == 'noise_freq':
                df = df[df.occ_frac > 0]
            if linestyles:
                ls = linestyles[i]
            else:
                ls = '-'
            df2 = df[(df.hr_model == hr_model) & (df.occ_transform == 'simplex_noise')]
            plot = sns.lineplot(
                data=df2, x=col, y='top1', marker='o', label=label, color=cmap(i), ax=ax, 
                errorbar='ci', err_style='bars', linestyle=ls
            )
        ax.legend(title=legend_name, loc='lower left')
        ax.set_ylabel('Accuracy', fontsize=20)
        ax.set_title(hr_model, fontsize=20)
        if col == 'noise_freq':
            ax.set_xscale('log')
            ax.set_xlabel('Occlusion frequency', fontsize=20)
        else:
            ax.set_xlabel('Occlusion fraction', fontsize=20)
    if save_name is None:
        save_name = name
    fig.savefig(f'out/sample_models_compare_{col}_{save_name}.png', dpi=100)
    plt.show()
    
def ushape_ex_human():
    fig, axs = plt.subplots(1, 3)
    fig.set_size_inches(10, 3)
    freqs = [1, 8, 64]
    occ_frac = 0.75
    npix_occ = round(224**2 * occ_frac)
    for ax, freq in zip(axs, freqs):
        im_name = f'ILSVRC2012_val_00011742_simplex_noise_{freq}_{npix_occ}.png'
        im = Image.open(f'../data/human_occ_exp/hexp1.1/simplex_noise_{freq}_{npix_occ}/images/{im_name}')
        ax.set_title(f'Frequency = {freq}')
        ax.imshow(im)
        ax.axis('off')
    plt.savefig('out/ushape_ex_human.png')
    plt.show()

def human_occ_acc_counts(img_df, name, save_name):
    fig, axs = plt.subplots(1, 3, constrained_layout=True)
    fig.set_size_inches(7 * 3, 5)
    suptitle = f'Occluded accuracy by subject ({name})'
    fig.suptitle(suptitle, fontsize=15)
    occ_frac_ranges = [(0.125, 0.875), (0.125, 0.25), (0.75, 0.875)]
    occ_frac_names = ['All occlusion fractions', 'Occlusion fraction ≤ 0.25', 'Occlusion fraction ≥ 0.75']
    for ax, occ_frac_range, occ_frac_name, in zip(axs, occ_frac_ranges, occ_frac_names):
        occ_frac_lo, occ_frac_hi = occ_frac_range
        df = img_df[(occ_frac_lo <= img_df.occ_frac) & (img_df.occ_frac <= occ_frac_hi)]
        human_unocc_mean = df.groupby('user_id').apply(lambda df: df[df.occ_frac == 0].correct.mean()).sort_values()
        human_occ_mean = df.groupby('user_id').apply(lambda df: df[df.occ_frac > 0].correct.mean()).sort_values()
        sns.histplot(human_occ_mean, ax=ax)
        ax.set_title(occ_frac_name, fontsize=15)
        ax.set_xlabel('Occluded accuracy', fontsize=15)
        ax.set_ylabel('Number of subjects', fontsize=15)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 30)
    plt.savefig(f'out/human_mean_occ_acc_counts_{save_name}.png', dpi=100)
    plt.show()

def human_acc_occ_vs_unocc_scatterplot(df, name, save_name, occ_frac_range=[0.125, 0.875], occ_frac_name=None):
    occ_frac_lo, occ_frac_hi = occ_frac_range
    occ_df = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac >= occ_frac_lo) & (df.occ_frac <= occ_frac_hi)]
    mean_occ_acc = occ_df.groupby('user_id').top1.mean().sort_values(ascending=False)
    unocc_df = df[(df.occ_transform == 'simplex_noise') & (df.occ_frac == 0)]
    unocc_acc = unocc_df.groupby('user_id').top1.mean().sort_values(ascending=False)
    # get the colormap
    c = 'green'
    # instantiate graphs
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    fig.suptitle(f'Occluded vs. unoccluded accuracy by subject ({name})', fontsize=15)
    # draw each graph
    # draw points on graph
    sns.scatterplot(x=unocc_acc, y=mean_occ_acc, color=c, marker='o', ax=ax)
    # draw lines on one of the graphs
    lr = LinearRegression(fit_intercept=True)
    lr.fit(np.array([unocc_acc]).T, mean_occ_acc)
    m, b = lr.coef_[0], lr.intercept_
    ax.plot([0, 1], [b, m + b], linestyle='--', color='red')
    ax.plot([0, 1], [0, 1], linestyle='--', color='black')
    # full x and y range
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Unoccluded accuracy', fontsize=15)
    ax.set_ylabel('Occluded accuracy', fontsize=15)
    fig.tight_layout()
    plt.savefig(f'out/human_acc_occ_vs_unocc_scatterplot_{save_name}.png', dpi=100)
    plt.show()
    
def empty_to_simplex_noise(df, occ_fracs, noise_freqs):
    # turn empty transform results into simplex_noise with occ_frac == 0 at all frequencies.
    # this makes graphing easier. then delete the empty transform, so only simplex_noise remain
    assert sorted(df.occ_frac.unique()) == occ_fracs
    assert sorted(df.noise_freq.dropna().unique()) == noise_freqs
    empty_df = df[df.occ_transform == 'empty']
    for noise_freq in noise_freqs:
        simplex_noise_df = empty_df.copy()
        simplex_noise_df.occ_transform = 'simplex_noise'
        simplex_noise_df.occ_frac = 0
        simplex_noise_df.noise_freq = noise_freq
        df = pd.concat([df, simplex_noise_df], axis=0)
    df = df[df.occ_transform == 'simplex_noise']
    return df

In [ ]:
# load human response data
none_human_hdf = load_hdf('../results/hexp1.1/human/human_responses.csv')
ddnm_human_hdf = load_hdf('../results/hexp1.1/human/human_responses_ddnm_1000.csv')
interp_human_hdf = load_hdf('../results/hexp1.1/human/human_responses_interp.csv')

print(len(none_human_hdf), len(ddnm_human_hdf), len(interp_human_hdf))

In [ ]:
# plot human no inpainting
single_acc_vs_occ_frac(df=none_human_hdf, name='humans, no inpainting', save_name='humans')
single_acc_vs_freq(df=none_human_hdf, name='humans, no inpainting', save_name='humans')
# human_acc_occ_vs_unocc_scatterplot(df=subj_df, name='no inpainting', save_name='humans')

# plot human interp inpainting
single_acc_vs_occ_frac(df=interp_human_hdf, name='humans, interp. inpainting', save_name='humans_interp')
single_acc_vs_freq(df=interp_human_hdf, name='humans, interp. inpainting', save_name='humans_interp')
# human_acc_occ_vs_unocc_scatterplot(df=isubj_df, name='interp. inpainting', save_name='humans_interp')

# plot human ddnm inpainting
single_acc_vs_occ_frac(df=ddnm_human_hdf, name='humans, DDNM inpainting', save_name='humans_ddnm')
single_acc_vs_freq(df=ddnm_human_hdf, name='humans, DDNM inpainting', save_name='humans_ddnm')
# human_acc_occ_vs_unocc_scatterplot(df=isubj_df, name='DDNM inpainting', save_name='humans_ddnm')

In [ ]:
# other human plots
inpaint_labels = [
    'None',
    'Interpolation',
    'DDNM',
]

inpaint_dfs = [
    none_human_hdf,
    interp_human_hdf,
    ddnm_human_hdf,
]

single_compare(col='occ_frac', labels=inpaint_labels, dfs=inpaint_dfs, name='humans', save_name='humans_inpaint', legend_name='Inpainter')
single_compare(col='noise_freq', labels=inpaint_labels, dfs=inpaint_dfs, name='humans', save_name='humans_inpaint', legend_name='Inpainter')

ushape_ex_human()

In [ ]:
# load model results on human experiment dataset
if False:  # making DF is slow operation so do only once
    model_hdf = mk_df('hexp1.1', include_ims=True)

model_hdf['hr_model'] = model_hdf.model.map(model_to_hr_model)
model_hdf = model_hdf.rename(columns={'correct': 'top1'})
# for models, keep only subset of image that humans saw in experiments
human_ims = set(none_human_hdf.filename.unique())
model_hdf = model_hdf[model_hdf.filename.isin(human_ims)]
# replace empty transforms with simplex noise for easier analysis
model_hdf = empty_to_simplex_noise(df=model_hdf, occ_fracs=hocc_fracs, noise_freqs=hnoise_freqs)

# separate by inpaint / train method
pt_model_hdf = model_hdf[(model_hdf.inpaint_model_precomp.isna()) & (model_hdf.checkpoint == 'pretrained')]
interp_model_hdf = model_hdf[(model_hdf.inpaint_model_precomp == 'interp') & (model_hdf.checkpoint == 'pretrained')]
deepfill_model_hdf = model_hdf[(model_hdf.inpaint_model_precomp == 'deepfill') & (model_hdf.checkpoint == 'pretrained')]
ddnm_100_model_hdf = model_hdf[(model_hdf.inpaint_model_precomp == 'ddnm_0.90_100') & (model_hdf.checkpoint == 'pretrained')]
ddnm_1000_model_hdf = model_hdf[(model_hdf.inpaint_model_precomp == 'ddnm_0.90_1000') & (model_hdf.checkpoint == 'pretrained')]
ft_model_hdf = model_hdf[(model_hdf.inpaint_model_precomp.isna()) & (model_hdf.checkpoint != 'pretrained')]

model_hdf_labels = [
    'None',
    'Interpolation',
    'DeepFill',
    'DDNM, T = 100',
    'DDNM, T = 1000',
    'None (finetuned)',
]

model_hdfs = [
    pt_model_hdf,
    interp_model_hdf,
    deepfill_model_hdf,
    ddnm_100_model_hdf,
    ddnm_1000_model_hdf,
    ft_model_hdf,
]

In [ ]:
for label, model_hdf_ in zip(model_hdf_labels, model_hdfs):
    print(label, len(model_hdf_) / hnocc / 200)

In [ ]:
linestyles = ['-'] * 5 + [':']
sample_models_compare(col='occ_frac', labels=model_hdf_labels, dfs=model_hdfs, save_name='inpaint', legend_name='Inpainter', linestyles=linestyles)
sample_models_compare(col='noise_freq', labels=model_hdf_labels, dfs=model_hdfs, save_name='inpaint', legend_name='Inpainter', linestyles=linestyles)

In [ ]:
def model_human_lineplot(
    human_dfs, human_labels, model_dfs, model_labels, 
    name, save_name=None, occ_frac_range=None, freq_range=None,
):
    cmap = plt.cm.get_cmap('Set2')

    fig, ax = plt.subplots(constrained_layout=True)
    fig.set_size_inches(14, 6)

    # only keep models in all dfs
    hr_models_set = set(model_dfs[0].hr_model.unique())
    for df in model_dfs:
        hr_models_set = hr_models_set.intersection(set(df.hr_model.unique()))
    for i, (label, df) in enumerate(zip(model_labels, model_dfs)):
        df = df.copy()
        df = df[df.hr_model.isin(hr_models_set)]
        df = df[df.occ_frac > 0]
        if occ_frac_range:
            lo, hi = occ_frac_range
            df = df[(lo <= df.occ_frac) & (df.occ_frac <= hi)]
        if freq_range:
            lo, hi = freq_range
            df = df[(lo <= df.noise_freq) & (df.noise_freq <= hi)]
        hr_models_sorted = df.groupby('hr_model').top1.mean().sort_values(ascending=False).index
        df['hr_model'] = pd.Categorical(df['hr_model'], categories=hr_models_sorted, ordered=True)
        sns.lineplot(data=df, x='hr_model', y='top1', ax=ax, errorbar='ci', err_style='bars', marker='o', label=label, c=cmap(i))

    for i, (label, df) in enumerate(zip(human_labels, human_dfs)):
        df = df.copy()
        df = df[df.occ_frac > 0]
        if occ_frac_range:
            lo, hi = occ_frac_range
            df = df[(lo <= df.occ_frac) & (df.occ_frac <= hi)]
        if freq_range:
            lo, hi = freq_range
            df = df[(lo <= df.noise_freq) & (df.noise_freq <= hi)]
        top1 = df.top1.mean()
        ax.axhline(y=top1, linestyle='--', label=label, c=cmap(i))

    ax.legend(title='')
    ax.set_xlabel('Model', fontsize=15)
    ax.set_ylabel('Accuracy', fontsize=15)
    title_extra_text = []
    title = f'Occluded accuracy for models and humans ({name})'
    ax.set_title(title, fontsize=15)
    plt.xticks(rotation=90)
    if not save_name:
        save_name = name
    fig.savefig(f'out/model_human_lineplot_{save_name}.png')
    plt.show()
    
model_dfs = [pt_model_hdf, ddnm_1000_model_hdf, ft_model_hdf]
model_labels = ['Models, pretrained', 'Models, DDNM-inpainted', 'Models, finetuned']

human_dfs = [none_human_hdf, ddnm_human_hdf]
human_labels = ['Humans', 'Humans, DDNM-inpainted']

In [ ]:
model_human_lineplot(
    human_dfs=human_dfs, human_labels=human_labels, model_dfs=model_dfs, model_labels=model_labels,
    occ_frac_range=[0.125, 0.25], name='occlusion fraction ≤ 0.25', save_name='low_occ_frac',
)
model_human_lineplot(
    human_dfs=human_dfs, human_labels=human_labels, model_dfs=model_dfs, model_labels=model_labels,
    occ_frac_range=[0.75, 0.875], name='occlusion fraction ≥ 0.75', save_name='high_occ_frac',
)
model_human_lineplot(
    human_dfs=human_dfs, human_labels=human_labels, model_dfs=model_dfs, model_labels=model_labels,
    freq_range=[1, 2], name='occlusion frequency ≤ 2', save_name='low_freq',
)
model_human_lineplot(
    human_dfs=human_dfs, human_labels=human_labels, model_dfs=model_dfs, model_labels=model_labels,
    freq_range=[8, 16], name='8 ≤ occlusion frequency ≤ 16', save_name='med_freq'
)
model_human_lineplot(
    human_dfs=human_dfs, human_labels=human_labels, model_dfs=model_dfs, model_labels=model_labels,
    freq_range=[64, 256], name='occlusion frequency ≥ 64', save_name='high_freq',
)

### Summary Tables ###

In [ ]:
# summary values

def calc_mean_acc(df, hr_model=None, occ_frac_range=None, freq_range=None, unoccluded=False):
    if hr_model:
        df = df[df.hr_model == hr_model]
    if unoccluded:
        df = df[df.occ_frac == 0]
    else: 
        df = df[df.occ_frac > 0]
    if occ_frac_range:
        lo, hi = occ_frac_range
        df = df[(lo <= df.occ_frac) & (df.occ_frac <= hi)]
    if freq_range:
        lo, hi = freq_range
        df = df[(lo <= df.noise_freq) & (df.noise_freq <= hi)]
    acc = df.top1.mean()
    return acc

subset_names = [
    'Unoccluded',
    'Occluded',
    'f ≤ 0.25',
    '0.375 ≤ f ≤ 0.625',
    'f ≥ 0.75',
    'ν ≤ 4',
    '8 ≤ ν ≤ 32',
    'ν ≥ 64',
]
subsets = [
    {'unoccluded': True},
    {},
    {'occ_frac_range': [0.125, 0.25]},
    {'occ_frac_range': [0.375, 0.625]},
    {'occ_frac_range': [0.75, 0.875]},
    {'freq_range': [1, 4]},
    {'freq_range': [8, 32]},
    {'freq_range': [64, 256]},
]

all_hr_models = model_to_hr_model.values()
sum_dfs = []
for df in [pt_model_hdf, ft_model_hdf, ddnm_1000_model_hdf, interp_model_hdf]:
    accs = []
    for hr_model in all_hr_models:
        model_accs = []
        for subset_name, subset in zip(subset_names, subsets):
            acc = calc_mean_acc(df=df, hr_model=hr_model, **subset)
            acc = np.round(acc, 3)
            model_accs.append(acc)
        accs.append(model_accs) 
    sum_df = pd.DataFrame(accs, columns=subset_names, index=all_hr_models)
    sum_df = sum_df.dropna()
    sum_dfs.append(sum_df)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    for sum_df in sum_dfs:
        print('summary df:')
        print(sum_df.to_latex())

### Scratch ###

In [ ]:
df2[(df2.occ_frac == 0.875) & (df2.noise_freq == 1)].top1, df2[(df2.occ_frac == 0.875) & (df2.noise_freq == 256)].top1

In [ ]:
gt_hdf = pd.read_csv('../data/human_occ_exp/hexp1.1/gt.csv')
im_sets = gt_hdf.groupby(gt_hdf.task_id.apply(lambda e: int(e[0]))).apply(lambda s: sorted(s.filename.unique().tolist()))

In [ ]:
results_loc = Path('../results/hexp1.1/eval')
accs_lst = []
models = []
for subexp_loc in tqdm(list(results_loc.glob('*'))):
    with open(subexp_loc / 'cfg.json', 'r') as f:
        cfg = json.load(f) 
    model = cfg['model']
    pd_df = pd.read_csv(subexp_loc / 'pred.csv', index_col=0)
    if len(pd_df) < 400:
        print(len(pd_df), cfg)
        continue
    accs = []
    for i in range(4):
        set_df = pd_df.loc[im_sets[i]]
        set_df['gt_idx'] = gt_df.loc[set_df.index].gt_idx
        set_df['correct'] = set_df.gt_idx == set_df.pd_idx
        acc = set_df.correct.mean()
        accs.append(acc)
    accs_lst.append(accs)
    models.append(model)
accs_lst = np.array(accs_lst)
sets_df = pd.DataFrame(accs_lst)
sets_df.index = models
sets_df

In [ ]:
for fn in human_hdf.filename.unique():
    h = human_hdf[(human_hdf.filename == fn) & (human_hdf.occ_frac == 0.75) & ()]
    h = ddnm_human_hdf[(ddnm_human_hdf.filename == fn) & (ddnm_human_hdf.occ_frac == 0.75)]
        print(fn)

In [ ]:
# quick calculations
interp_human_hdf[(interp_human_hdf.occ_frac == 0) & (interp_human_hdf.noise_freq == 64)].top1.mean()
for df in [human_hdf, interp_human_hdf, ddnm_human_hdf]:
    x = df[(df.noise_freq == 8)].top1.mean()
    print(f'{x * 100:.1f}')

In [ ]:
model_hdf[(model_hdf.inpaint_model_precomp == 'interp') & (model_hdf.checkpoint == 'pretrained') & (model_hdf.hr_model == 'CLIP-B/16 (WIT)')
        & (model_hdf.noise_freq == 8) & (model_hdf.occ_frac > 0)].top1.mean()

In [ ]:
for df in [none_human_hdf, interp_human_hdf, ddnm_human_hdf]:
    x = df[(df.occ_frac > 0) & (df.noise_freq == 8)].top1.mean()
    print(x)

In [ ]:
pt_df[(pt_df.hr_model == 'ResNet-18 (IN1k)') & (pt_df.occ_frac > 0)].top1.mean(), ft_df[(ft_df.hr_model == 'ResNet-18 (IN1k)') & (ft_df.occ_frac > 0)].top1.mean()

In [ ]:
df = ddnm_1000_model_hdf
df[(df.hr_model == 'CLIP-B/16 (WIT)') & (df.occ_frac > 0) & (df.noise_freq == 8)].top1.mean()

In [ ]:
df = interp_model_hdf
df[(df.hr_model == 'CLIP-B/16 (WIT)') & (df.occ_frac > 0) & (df.noise_freq == 8)].top1.mean()